# 30 对 Vue/San 组件生产后的问题、修复与研究启示

**记录日期：** 2026-09-04  
**数据批次：** `design_matrix_2026-09-02_batch_30.json`  
**数据范围：** simple、medium、complex 各 10 对，共 30 对 Vue/San 组件  
**记录目的：** 汇总组件批量生产完成后，在逐项人工对照检查中暴露的问题、根因、解决办法与残余风险，为论文中的数据质量控制、迁移难点、实验结果和有效性威胁分析保留可追溯材料。

> 本文中的“人工确认通过”表示用户在 Vue/San 手工测试页中检查了初始展示和主要交互。它不等同于自动化回归测试、跨浏览器测试或像素级视觉等价验证。

## 1. 本批组件清单与验证结论

| 复杂度 | 组件（按设计矩阵与人工检查顺序） |
|---|---|
| Simple | `PasswordStrengthMeter`、`VolumeControl`、`InlineRename`、`QueueTicket`、`SearchClearField`、`ReactionBar`、`UnitConverter`、`SortCycleButton`、`PasswordVisibilityField`、`BudgetThreshold` |
| Medium | `TransferList`、`PollResults`、`RecipeScaler`、`BatchRenamer`、`ShippingEstimator`、`AttendanceRoster`、`PlaybackQueue`、`SurveyBranchForm`、`SeatBookingMap`、`BreadcrumbNavigator` |
| Complex | `DocumentApprovalFlow`、`CapacityPlanningBoard`、`LocalizationWorkspace`、`SupportTicketConsole`、`TripItineraryPlanner`、`SchemaMappingWorkbench`、`ExperimentRolloutConsole`、`InventoryBatchManager`、`IncidentResponseBoard`、`GradebookMatrix` |

最终状态：30 对组件均已按顺序完成人工确认；测试页最终保留的组件为 `GradebookMatrix`。其中 6 个组件在生产完成后需要源码修复。

In [ ]:
components = {
    'simple': [
        'PasswordStrengthMeter', 'VolumeControl', 'InlineRename',
        'QueueTicket', 'SearchClearField', 'ReactionBar', 'UnitConverter',
        'SortCycleButton', 'PasswordVisibilityField', 'BudgetThreshold'
    ],
    'medium': [
        'TransferList', 'PollResults', 'RecipeScaler', 'BatchRenamer',
        'ShippingEstimator', 'AttendanceRoster', 'PlaybackQueue',
        'SurveyBranchForm', 'SeatBookingMap', 'BreadcrumbNavigator'
    ],
    'complex': [
        'DocumentApprovalFlow', 'CapacityPlanningBoard',
        'LocalizationWorkspace', 'SupportTicketConsole',
        'TripItineraryPlanner', 'SchemaMappingWorkbench',
        'ExperimentRolloutConsole', 'InventoryBatchManager',
        'IncidentResponseBoard', 'GradebookMatrix'
    ],
}

print('组件对总数:', sum(len(names) for names in components.values()))
print('复杂度分布:', {level: len(names) for level, names in components.items()})

## 2. 人工检查方法与证据边界

本批采用串行人工验证：每次只把一对组件配置到 `tests/manual/vue-test-runner.html` 和 `tests/manual/san-test-runner.html`，用户对照检查后回复确认，再切换到下一个组件。发现问题时停止推进，先修复当前组件并复核。检查关注以下可观察行为：

1. props 和初始化数据是否显示；
2. 默认选中态、动态 class 和禁用态是否正确；
3. 点击、输入、筛选、增删和状态推进是否立即更新界面；
4. 派生统计、列表数量和状态文案是否与数据变化同步；
5. Vue/San 两端是否存在遮挡、闪烁或页面无法加载。

该方法能发现静态校验遗漏的运行时与视觉问题，但当前证据主要是人工确认记录，尚未保存截图、浏览器控制台日志和可重复执行的交互脚本。

## 3. 生产后问题总览

### 3.1 已解决的生产后缺陷

| 编号 | 组件/框架 | 人工检查现象 | 根因 | 最终解决办法 | 状态 |
|---|---|---|---|---|---|
| D1 | `RecipeScaler` / San | 点击“切换单位”后单位不立即变化，增减份数后才刷新 | `formatAmount()` 在方法内部读取 `unit`，模板调用没有显式依赖 `unit` | 模板改为 `formatAmount(value, unit)`，方法使用传入的 `unit` | 已解决并人工确认 |
| D2 | `SeatBookingMap` / San | 点击座位后数量变化，但座位没有及时变色 | `seatClass()` 隐式读取 `selectedIds`，San 未建立模板响应式依赖 | 改为 `seatClass(seat, selectedIds)` | 已解决并人工确认 |
| D3 | `TripItineraryPlanner` / Vue、San | 行程列表、按钮和新增表单发生遮挡 | 固定网格列宽超过组件可用宽度，缺少换行与收缩约束 | 两端同步改为带 `min-width: 0` 和 `flex-wrap` 的弹性布局 | 已解决并人工确认 |
| D4 | `TripItineraryPlanner` / San | 增删行程后日期旁的项目数量不立即更新；冲突样式也存在同类风险 | `dayCount()` 与 `isConflict()` 的模板调用未显式包含 `items`/`conflictIds` | 改为 `dayCount(day.id, items)` 和 `isConflict(item.id, conflictIds)` | 已解决并人工确认 |
| D5 | `SchemaMappingWorkbench` / San | “转换规则”下拉框点开后闪烁，难以操作 | 映射数组更新时循环节点缺少稳定身份，San 重建了包含 `<select>` 的行 | 为映射循环增加 `trackby mapping.id` | 已解决并人工确认 |
| D6 | `ExperimentRolloutConsole` / San | 点击“设为胜出”后胜出行没有及时显示绿色，顶部状态也可能滞后 | `rowClass()` 和 `statusText()` 隐式读取响应式状态 | 改为 `rowClass(row, winnerId, leaderId)` 与 `statusText(status)` | 已解决并人工确认 |
| D7 | `InventoryBatchManager` / San | 单选或“全选结果”后，复选框和行背景没有及时变化 | `isSelected()` 与 `batchRowClass()` 隐式读取 `selectedIds` | 两个模板调用均显式传入 `selectedIds` | 已解决并人工确认 |

### 3.2 修复过程中出现的回归

| 编号 | 组件/框架 | 现象 | 原因与处理 | 状态 |
|---|---|---|---|---|
| R1 | `TripItineraryPlanner` / Vue | 第一次处理遮挡后，Vue 测试页无法加载 | 初版响应式 CSS 补丁与当前旧版 `http-vue-loader` 的 scoped-style 处理不兼容；最终去掉 `@media`，使用无需媒体查询的自然换行布局 | 已解决并人工确认 |

### 3.3 经核对属于预期规则的现象

`ExperimentRolloutConsole` 调整单个流量滑块后，“开始”按钮不可用并非业务逻辑错误。组件要求所有变体的流量总和等于 100%；只改一个滑块会破坏该约束。该现象暴露的是**约束说明不够明显**：后续可增加禁用原因、自动互补分配或总量校正操作，降低用户误判。

In [ ]:
from collections import Counter

primary_defects = [
    {'id': 'D1', 'component': 'RecipeScaler', 'level': 'medium', 'scope': 'san', 'cause': 'hidden_reactive_dependency'},
    {'id': 'D2', 'component': 'SeatBookingMap', 'level': 'medium', 'scope': 'san', 'cause': 'hidden_reactive_dependency'},
    {'id': 'D3', 'component': 'TripItineraryPlanner', 'level': 'complex', 'scope': 'shared', 'cause': 'layout_overflow'},
    {'id': 'D4', 'component': 'TripItineraryPlanner', 'level': 'complex', 'scope': 'san', 'cause': 'hidden_reactive_dependency'},
    {'id': 'D5', 'component': 'SchemaMappingWorkbench', 'level': 'complex', 'scope': 'san', 'cause': 'missing_list_identity'},
    {'id': 'D6', 'component': 'ExperimentRolloutConsole', 'level': 'complex', 'scope': 'san', 'cause': 'hidden_reactive_dependency'},
    {'id': 'D7', 'component': 'InventoryBatchManager', 'level': 'complex', 'scope': 'san', 'cause': 'hidden_reactive_dependency'},
]

total_pairs = sum(len(names) for names in components.values())
affected_components = {item['component'] for item in primary_defects}
affected_by_level = Counter(
    level for level, names in components.items()
    for name in names if name in affected_components
)

print('生产后缺陷记录:', len(primary_defects))
print('受影响组件:', len(affected_components))
print('受影响组件比例: {:.1%}'.format(len(affected_components) / total_pairs))
print('受影响组件复杂度分布:', dict(affected_by_level))
print('缺陷根因分布:', dict(Counter(item['cause'] for item in primary_defects)))
print('San 特有缺陷占比: {:.1%}'.format(
    sum(item['scope'] == 'san' for item in primary_defects) / len(primary_defects)
))

## 4. 根因分析

### 4.1 最主要根因：San 模板无法感知方法内部的隐藏依赖

D1、D2、D4、D6、D7 具有同一结构：模板调用一个方法，但影响返回值的响应式变量只在方法内部通过 `this.data.get()` 读取，没有作为模板参数出现。状态本身已经成功更新，数量或其他 computed 甚至也会变化，但方法绑定的 DOM 不会立即重新求值，于是形成“点击有效、视觉反馈滞后”的假象。

这与普通 JavaScript 正确性不同。方法能够读到最新数据，不代表模板运行时能建立正确的依赖图。对迁移系统而言，转换 `this.xxx` 为 `this.data.get('xxx')` 只解决了访问语法，尚未解决**模板调用点的依赖可见性**。

稳定修复原则是：模板方法的返回值若依赖可变状态，应把该状态显式作为参数传入，或将逻辑改造成依赖清晰的 computed 数据。

### 4.2 列表身份是交互状态的一部分

D5 说明 `v-for :key` 不能只在 Vue 端保留。包含输入框、下拉框或焦点状态的循环节点若在 San 中缺少 `trackby`，不可变数组更新可能触发 DOM 行重建，造成下拉框关闭、光标丢失或闪烁。迁移规则应把 Vue 的稳定 key 显式翻译为 San 的 `trackby`。

### 4.3 视觉等价还受运行载体约束

D3 与 R1 表明，组件 CSS 不仅要在设计宽度下正确，还要兼容实际手工 runner 使用的加载器。固定网格对内容长度和容器宽度敏感；补丁中使用构建环境不支持的 CSS 处理路径，又会让视觉修复转化为加载失败。最终采用无 `@media` 的弹性换行，是当前运行环境下更稳妥的共同实现。

## 5. 可复用的修复模式

| 风险模式 | 修复前 | 修复后 |
|---|---|---|
| 格式化方法隐藏状态依赖 | `formatAmount(value)`，方法内部读取 `unit` | `formatAmount(value, unit)` |
| 动态 class 隐藏集合依赖 | `seatClass(seat)` | `seatClass(seat, selectedIds)` |
| 动态计数隐藏源数组依赖 | `dayCount(day.id)` | `dayCount(day.id, items)` |
| 动态状态样式隐藏多个依赖 | `rowClass(row)` | `rowClass(row, winnerId, leaderId)` |
| 循环交互节点无稳定身份 | `s-for="mapping in mappings"` | `s-for="mapping in mappings trackby mapping.id"` |
| 固定宽度网格溢出 | 固定列宽且不换行 | `min-width: 0`、弹性基准宽度和 `flex-wrap` |

这些模式适合转化为生成规则与静态门禁，而不是仅保留为单组件补丁。尤其应检查：模板方法实际读取了哪些响应式字段、调用点是否显式暴露这些字段、循环节点是否继承 Vue key 语义。

In [ ]:
from pathlib import Path

def find_project_root():
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / 'data' / 'datasets' / 'components').exists():
            return candidate
    raise FileNotFoundError('未找到项目根目录')

root = find_project_root()
checks = {
    'RecipeScaler 显式 unit 依赖': (
        'data/datasets/components/02_medium/RecipeScaler/san/RecipeScaler.san',
        ['formatAmount(item.amount, unit)', 'formatAmount(totalWeight, unit)']
    ),
    'SeatBookingMap 显式 selectedIds 依赖': (
        'data/datasets/components/02_medium/SeatBookingMap/san/SeatBookingMap.san',
        ['seatClass(seat, selectedIds)']
    ),
    'TripItineraryPlanner 显式列表依赖': (
        'data/datasets/components/03_complex/TripItineraryPlanner/san/TripItineraryPlanner.san',
        ['dayCount(day.id, items)', 'isConflict(item.id, conflictIds)', 'flex-wrap:wrap']
    ),
    'SchemaMappingWorkbench 稳定行身份': (
        'data/datasets/components/03_complex/SchemaMappingWorkbench/san/SchemaMappingWorkbench.san',
        ['trackby mapping.id']
    ),
    'ExperimentRolloutConsole 显式状态依赖': (
        'data/datasets/components/03_complex/ExperimentRolloutConsole/san/ExperimentRolloutConsole.san',
        ['statusText(status)', 'rowClass(row, winnerId, leaderId)']
    ),
    'InventoryBatchManager 显式 selectedIds 依赖': (
        'data/datasets/components/03_complex/InventoryBatchManager/san/InventoryBatchManager.san',
        ['isSelected(batch.id, selectedIds)', 'batchRowClass(batch, selectedIds)']
    ),
}

results = {}
for label, (relative_path, expected_fragments) in checks.items():
    source = (root / relative_path).read_text(encoding='utf-8')
    results[label] = all(fragment in source for fragment in expected_fragments)

for label, passed in results.items():
    print(('PASS' if passed else 'FAIL'), '-', label)

assert all(results.values()), '至少一个已知修复模式缺失'

## 6. 验证结果与可报告指标

- 样本规模：30 对 Vue/San 组件，simple、medium、complex 各 10 对。
- 串行人工检查覆盖率：100%（30/30）。
- 生产后发现缺陷记录：7 条，涉及 6 个组件。
- 受影响组件比例：20.0%（6/30）。
- 按复杂度统计受影响组件：simple 0/10，medium 2/10，complex 4/10。
- 7 条缺陷中，San 特有问题 6 条（85.7%），Vue/San 共有布局问题 1 条（14.3%）。
- 隐藏响应式依赖导致的缺陷 5 条，占全部生产后缺陷的 71.4%。
- 另记录修复过程回归 1 条、预期业务约束澄清 1 条；二者不计入生产后缺陷率。
- 所有组件最终均由用户人工确认，但尚无自动化回归证据。

这些数字仅描述当前有目的构造且经人工检查的批次，不能直接外推为一般 Vue→San 迁移任务的总体缺陷率。

## 7. 当前仍待工程化解决的问题

### 7.1 生成源与人工修复结果存在漂移

当前 `scripts/generate_dataset_batch_20260902.js` 仍保存修复前的组件规格，例如只传 `formatAmount(value)`、`seatClass(seat)`、`dayCount(day.id)`、`rowClass(row)` 和 `batchRowClass(batch)`，并保留缺少 `trackby` 的映射循环及原固定网格布局。若重新运行生成脚本，已修复的 `.san`/`.vue` 文件可能被覆盖并重新引入缺陷。

建议：把修复同步回生成规则或增加 San 专用模板覆盖，并添加“重新生成后运行回归检查”的门禁。

### 7.2 静态验证没有覆盖响应式依赖图

本批组件在生产阶段已通过静态验证，但 5 条隐藏依赖问题仍进入人工检查。后续校验器应分析模板方法调用与方法内部 `this.data.get()` 的关系：当返回值依赖的字段未出现在调用参数或 computed 依赖中时，报告潜在的视图滞后风险。

### 7.3 Vue key 到 San trackby 的迁移规则不完整

建议将 `v-for` 的 `:key` 映射为 San 的 `trackby`，并优先覆盖包含表单控件、焦点和局部 DOM 状态的循环列表。

### 7.4 人工结果未同步到结构化元数据

`data/datasets/features/migration_notes.json` 中本批条目的 `functional_test` 仍为 `static_validate_passed`，`visual_test` 仍为 `not_run`。这与实际完成的人工确认不一致。论文统计前应定义单独的 `manual_test` 记录，包含日期、组件、Vue 结果、San 结果、交互清单、发现问题、修复提交和复核结果，避免把静态通过、人工通过和自动化通过合并为一个状态。

### 7.5 缺少可重复的浏览器回归

建议为每对组件保存最小交互协议：初始数据可见、默认状态、一次核心点击、一次输入变化、一次派生数据变化和关键动态样式。Vue/San 执行相同动作后比较 DOM 状态、文本和截图，同时捕获控制台错误。

## 8. 可用于论文的归纳

### 8.1 数据质量控制结论

1. 文件存在、语法合法和 props 对齐只能证明样本具备静态完整性，不能证明目标框架中的交互等价。
2. Vue→San 迁移的关键难点不只在语法替换，还在两个框架建立响应式依赖图的方式不同。
3. 方法内部隐藏依赖会造成一种高迷惑性的缺陷：数据已改变，但部分视图要等到其他状态变化后才刷新。
4. 列表 key、焦点与表单控件状态属于行为语义，迁移时不能被视为可选的渲染优化。
5. 视觉修复需要在真实加载器和容器约束下验证，否则可能引入新的运行环境回归。
6. 人工串行检查适合发现未知问题，但要形成可靠实验结论，还需结构化证据和自动化复现。

### 8.2 可讨论的复杂度趋势

本批 simple 组件未发现需源码修复的问题，medium 有 2 个受影响组件，complex 有 4 个。该结果与“状态越多、列表越深、交互链越长，跨框架迁移风险越高”的预期一致，但每层只有 10 个样本，且样本是有目的设计而非随机抽样，因此只能作为探索性证据。

### 8.3 有效性威胁

- 人工测试路径可能未覆盖所有分支、边界输入、定时器和事件载荷。
- 测试环境使用 CDN 与旧版 `http-vue-loader`，结果可能受网络、缓存和加载器实现影响。
- 缺少修复前截图、控制台日志和自动化脚本，部分因果判断依赖修复过程记录。
- 同一批次由同一生产规则生成，缺陷并非独立同分布，比例不能直接代表外部项目。
- 当前生成脚本尚未吸收人工补丁，复现实验前必须固定代码版本并说明是否重新生成。

## 9. 后续行动建议

1. 将本批 6 个组件的修复同步到生成脚本或转换规则，确保重新生成结果稳定。
2. 为模板方法建立隐藏响应式依赖检查，并为 `v-for :key` → `s-for trackby` 增加转换与校验规则。
3. 为 30 对组件建立相同的最小浏览器交互脚本，保存 DOM 断言、控制台日志与截图。
4. 在元数据中新增独立的人工验证记录，不覆盖静态验证和自动化验证字段。
5. 论文实验中分别报告组件受影响率、缺陷条数、缺陷根因分布和修复后复核率，避免用单一“通过率”掩盖修复成本。

## 10. 相关项目文件

- 本批设计矩阵：`data/datasets/features/design_matrix_2026-09-02_batch_30.json`
- 数据集清单：`data/datasets/dataset_manifest.json`
- 迁移说明：`data/datasets/features/migration_notes.json`
- 批量生成脚本：`scripts/generate_dataset_batch_20260902.js`
- 数据生产规则：`skills/vue-san-dataset-production/SKILL.md`
- 人工测试配置规则：`skills/manual-runner-config/SKILL.md`
- Vue 测试页：`tests/manual/vue-test-runner.html`
- San 测试页：`tests/manual/san-test-runner.html`
- 上一批问题记录：`notebooks/01-data-exploration/2026-09-02_vue_san_batch_validation_issues.ipynb`